# RAG01｜遊戲機制實驗室｜建立專屬向量資料庫

**目標：** 讀入原創遊戲機制設計文件 → 分塊 → Embedding → FAISS 向量索引 → 輸出 `rag-data.zip`。

**資料說明：** `game_mechanics_manual.txt` 是為本作業創作的原創遊戲機制規格，共 26 個章節，涵蓋戰術射擊、類倖存者、節奏判定與三消戰鬥的明確規則、公式、測試案例及未定案邊界。文件隨此 Notebook 寫入，不須另外上傳。

**技術選擇：** 保留老師範例 `sentence-transformers`、`transformers`、`faiss-cpu` 與向量化流程。預設使用無須 Hugging Face 授權的 `intfloat/multilingual-e5-small`，降低助教重跑時的授權門檻；如果在 Colab Secret 配置 `HuggingFace`，可改用老師範例的 `google/embeddinggemma-300m`。模型及檢索前綴會記在 ZIP 的 `manifest.json` 供 RAG02 自動沿用。

**AI 協作聲明：** 此 Notebook 由生成式 AI 協助搭建程式結構、整理 Prompt 與安全下載檢查。程式需由本人於 Colab 執行、理解、測試並截圖；不可將尚未執行的程式視為已驗證成果。

## 1. 安裝老師的核心固定套件

請在 **Google Colab** 由上往下執行。首次執行會下載 Embedding 模型，需要連網。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 撰寫並載入原創文本

這一格會在 Colab 當前工作目錄建立真正的 `.txt` 文件；可在左側檔案瀏覽器檢查。

In [ ]:
from pathlib import Path
SOURCE_PATH = Path('/content/game_mechanics_manual.txt') if Path('/content').exists() else Path('game_mechanics_manual.txt')
ORIGINAL_DOCUMENT = '# 《遊戲機制實驗室》機制拆解與數值平衡手冊\n版本：2026-10-31 / 原創教學資料 v1.0\n用途：RAG 課堂示範用的原創遊戲機制規格。這是一份跨類型「機制設計實驗室」，不是某款已發售遊戲的官方攻略。以下數值為可重現的教學原型預設值；沒有寫明的規格一律視為未定案。不同原型的規則不可混用。\n\n## 01｜什麼是遊戲機制\n遊戲機制是玩家操作、系統狀態、規則判定與回饋之間的可重複關係。回答某個機制時，優先描述「玩家動作 → 前置條件 → 狀態轉換 → 數值變化 → 視覺或音效回饋 → 失敗與例外」。遊戲循環則描述玩家為了達成目標而反覆執行的一組機制；單一技能冷卻是機制，「戰鬥→獲取資源→升級→挑戰更強敵人」是循環。美術風格與世界觀本身不構成機制。\n\n## 02｜教學原型的名稱與邊界\n本手冊包含四個獨立原型：A《盾線行動》（俯視角戰術射擊）、B《群潮生存》（類倖存者自動攻擊）、C《節拍深淵》（音樂節奏判定）、D《符文對決》（三消轉換戰鬥）。四者僅共享「可測試、可觀測、可平衡」的設計原則，不共享貨幣、體力、等級、敵人或存檔。若問題未指明原型，先指出需要確認哪個原型，或分別比較，不能假設四者是同一款遊戲。\n\n## 03｜核心循環與目標\nA《盾線行動》：觀察威脅→決定走位與射擊→管理彈藥→消除威脅→抵達撤離點。關卡勝利必須完成主要目標並抵達撤離點，殺光敵人本身不算通關。B《群潮生存》：移動閃避→自動攻擊→收集經驗→選升級→在 10 分鐘結算存活。C《節拍深淵》：辨識節奏→擊打判定→維持連擊→結束曲目計算成績。D《符文對決》：交換相鄰符文→形成三連→結算攻防效果→輪到敵方行動→使敵方血量歸零。\n\n## 04｜狀態機與事件處理\n以 A 的玩家角色為例，主狀態為 Idle、Move、Attack、Reload、Stunned、Dead。Reload 中不能開火；Dead 是不可逆的本關終止狀態；Stunned 會中斷 Reload 並取消尚未轉移的備彈。攻擊按鍵事件先檢查 Dead、Stunned、Reload、射擊冷卻與彈匣子彈數。一次按鍵最多產生一發子彈。B 的一般敵人以 Spawn、Seek、Attack、Dead 為主要狀態；Dead 不會重新進入 Seek。\n\n## 05｜A：射擊與命中規則\nA 使用俯視角射擊，玩家初始生命 100 HP。步槍基本傷害 16，射速每秒 5 發，連射間隔 0.20 秒，最大射程 12 公尺。每發子彈至多命中一個目標，命中障礙物會停止；不含穿透。敵人護甲先減去 25% 的原始傷害，再從 HP 扣除，結果四捨五入到小數點後兩位。因此步槍命中有護甲敵人的傷害為 16×(1−0.25)=12。沒有爆擊系統；不可自行宣稱爆頭有倍率。\n\n## 06｜A：彈匣、換彈與彈藥預算\nA 的步槍彈匣可裝 24 發，出發時彈匣已滿，另外帶 72 發備彈，因此總共最多可使用 96 發。完整換彈需要 1.6 秒，採戰術換彈：彈匣中剩餘子彈保留，僅從備彈補到 24 發。沒有自動生成子彈機制。關卡內只有兩個固定彈藥補給箱，各提供 12 發；若全部取得，玩家可用步槍彈藥總數最多 120 發。補給箱只能領取一次；未拾取的子彈不會自動加入背包。\n\n## 07｜A：關卡傷害預算與 Boss\nA 的測試關卡共有 6 名一般敵人，每人 60 HP 且沒有護甲；Boss 有 320 HP、25% 護甲。理論上全部命中且只用步槍時，一般敵人每人至少需要 ceil(60/16)=4 發，共 24 發；Boss 每發 12 傷害，至少需要 ceil(320/12)=27 發；不考慮其他限制的理論下限共 51 發。這不是實際通關保證，因為有彈道失誤、掩體遮擋與擊發時機。Boss 生命降到 160 HP 以下時進入第二階段，移動速度提升 20%，護甲不變，沒有無敵階段。\n\n## 08｜A：難度調整與缺彈處理\nA 的 Easy 與 Normal 都有相同出發彈藥，但 Normal 敵人移動速度較 Easy 提升 15%，且補給箱改放在較難到達的側路。若自動化測試在 Normal 打到 Boss 剩 86/320 HP 時耗盡子彈，先量測命中率、射擊障礙物比例、補給箱拾取率與 Boss 受傷事件，再判斷要修正 AI、資源配置或敵方血量，不可以直接把「補充 86 HP 所需子彈」當作所有玩家固定缺彈量。若要量化最少所需 Boss 命中數，ceil(86/12)=8 發；實際上至少需要 8 次有效命中，可能需要更多射擊。\n\n## 09｜A：掩體、預警與回饋\nA 的掩體可阻擋直線彈道，但不降低靠近後的命中傷害。敵人射擊前會有 0.35 秒的準備動畫，玩家受到傷害時顯示方向指示、減血數字與短暫受擊閃爍。需要用「場景遮擋測試」驗證碰撞層，不能只靠視覺特效推斷玩家真的沒受傷。正常模式沒有自動恢復生命機制。視覺、音效回饋需在傷害判定後才觸發。\n\n## 10｜B：群潮生存的移動與自動攻擊\nB 的角色只能移動與選升級，基礎武器自動尋找半徑 7 公尺內最近的敵人，每 0.8 秒發射一次，基礎傷害 10。若範圍內沒有敵人就不會發射，也不儲存射擊次數。玩家移動速度為每秒 4 公尺，碰觸敵人每次受到 8 傷害；同一敵人對同一玩家碰觸後有 1 秒再次傷害冷卻。升級後的額外武器另依各自冷卻運作。\n\n## 11｜B：敵人 AI 與物件池\nB 的一般敵人使用 Seek（追逐玩家）與 Separation（避免同類堆疊）兩個轉向力，實際位移向量為 normalize(1.0×Seek+0.35×Separation)×移動速度。敵人不具備尋路能力，因此不應宣稱會繞過牆壁。生成由 SpawnDirector 控制，系統每秒最多建立 3 個敵人實體，且同場最多 60 名活敵人。死亡敵人回收到 Object Pool，不在死亡時立即銷毀場景物件。回收前需清除狀態、碰撞與特效旗標，防止重複受傷事件。\n\n## 12｜B：波次與生成公平性\nB 每 60 秒提升一次生成速率；基礎每秒嘗試生成 0.5 名敵人，經過 n 分鐘後目標生成率為 min(0.5+0.15n, 2.0) 名/秒。生成位置須在玩家鏡頭可見區域外、距離玩家至少 5 公尺，並通過障礙物重疊檢查。如果找不到合法位置，延後此次生成，不得在玩家腳下強制生成。活敵人數已達 60 時直接跳過該次生成。\n\n## 13｜B：經驗、等級與局外養成\nB 一般敵人掉落 1 點經驗。升至 2 級需累積 8 XP，升至 3 級需累積 20 XP，升至 4 級需累積 36 XP；此表列的是「總累積 XP 門檻」，不是每級額外需求。升級時從三個隨機候選中挑一個：攻擊力 +15%、冷卻時間 −10%、最大 HP +20；同一候選最多選三次。局外永久升級只影響起始最大 HP，每級 +5、最多四級，不改變掉落機率。結算時死亡或存活都可獲得金幣，數量依存活秒數計算，公式為 floor(秒數/30)。\n\n## 14｜C：節奏遊戲的輸入時間\nC 的音符在預定節點判定，時間誤差定義為 |玩家按鍵時間 − 音符理想時間|。Perfect：誤差 ≤45 毫秒；Great：45 毫秒 < 誤差 ≤110 毫秒；Miss：誤差 >110 毫秒或未輸入。這些區間互斥，因此誤差 45 毫秒為 Perfect，110 毫秒為 Great，111 毫秒為 Miss。每一音符只會判定一次；同一按鍵不可同時領取兩個音符的分數。\n\n## 15｜C：分數、連擊與延遲校正\nC 的 Perfect 得 1000 分、Great 得 600 分、Miss 得 0 分。Perfect 和 Great 都令 Combo +1，Miss 會使 Combo 歸零。結算分數為所有音符原始分數相加，不使用 Combo 倍率；Combo 僅用於視覺回饋與成就。延遲校正值會加在輸入時間戳上，範圍 −200 至 +200 毫秒，預設 0；玩家需透過測試音校正，不應把延遲補償當成擴大判定窗。\n\n## 16｜D：三消移動與交換合法性\nD 使用 8×8 方格。玩家只能交換上下左右相鄰的兩格，不能對角交換。交換後若沒有形成水平或垂直連續至少 3 個同色符文，兩格會彈回原位且不消耗回合；有效交換消耗 1 回合。一次交換最多產生一個玩家回合結算，但連鎖下落的所有消除都包含在該回合內。棋盤開局不得有現成三連，且必須至少有一步有效交換。\n\n## 17｜D：傷害、護盾與連鎖\nD 的紅色符文每消除 1 顆造成 4 傷害，藍色符文每消除 1 顆增加 3 護盾，綠色符文每消除 1 顆恢復 2 HP（不得超過最大 HP）。一次消除 4 顆紅色符文造成 16 傷害，不額外加成；5 顆也沒有特殊炸彈。連鎖倍率只作用於紅色傷害：第一次消除 ×1，第一次下落連鎖 ×1.25，第二次下落連鎖 ×1.5；結果逐次四捨五入到整數。藍色護盾與綠色回血不受連鎖倍率影響。\n\n## 18｜D：敵方行動與回合順序\nD 的結算順序固定是：檢查交換合法性 → 消除與連鎖全部結束 → 玩家符文效果結算 → 判定敵人是否死亡 → 若未死亡，敵人反擊一次 → 確認勝敗。敵人的基礎反擊傷害為 12；先扣護盾，再扣 HP。若消除使敵人生命降至 0，敵人不反擊。護盾不會自動清空，會延續到下一回合。這是回合制遊戲，不使用即時射擊的命中率概念。\n\n## 19｜跨類型：風險報酬與選擇\n有效的風險報酬選擇需同時具備可見風險、玩家可控制的行動與足以理解的結果。A 的側路彈藥箱是「多走危險路線換更多彈藥」；B 的高密度經驗群是「接近敵人換更快升級」；C 的密集音符區考驗準確性但沒有獨立風險押注系統；D 的有效交換為回合資源，每次都必須與敵方反擊權衡。不能把純隨機懲罰描述成有意義的策略選擇。\n\n## 20｜跨類型：難度曲線與可讀性\n難度不是只提高敵人 HP；可分成認知負荷、執行精度、資源壓力與策略深度。A 主要考驗射擊命中率、掩體與彈藥；B 考驗走位與生成密度；C 考驗時序準確性；D 考驗交換規劃與護盾管理。調高難度時應一次只變更一類主要因素，並保留能觀察到的失敗原因。所有原型都要求失敗後能回顧基本指標。\n\n## 21｜跨類型：UI 與可及性\n每個狀態應有對應回饋。A 要顯示剩餘彈匣與備彈、重裝進度；B 要顯示 XP 進度、玩家 HP、剩餘時間；C 要顯示音符判定與延遲校正；D 要顯示剩餘回合、HP 與護盾。關鍵狀態不可只使用顏色傳達，還須有圖示或文字。使用者調整 UI 縮放不應改變遊戲判定視窗或傷害。\n\n## 22｜跨類型：平衡流程\n數值平衡依序進行：定義目標體驗與可接受區間、建立理論計算、用機器測試大量跑局、記錄異常分布、進行真人測試、再調參。A 的關卡分析需分開統計射擊數、有效命中、彈藥箱拾取、死亡原因與剩餘 Boss HP；B 記錄每分鐘生存率與同屏敵人；C 記錄判定分布及校正前後的偏移；D 記錄平均回合、有效交換率與護盾溢出量。平均值不能取代分位數與失敗案例。\n\n## 23｜跨類型：事件紀錄與可重現測試\n所有機制的重要事件至少記錄 timestamp、run_id、mechanic_id、state_before、action、state_after、value_delta 與 result。隨機系統固定 RNG seed 以重跑測試。A 的命中需保存 hit_target、damage_after_armor、obstacle_blocked；B 的生成需記錄 valid_spawn、active_enemy_count；C 需記錄 timing_offset_ms、judgment；D 需記錄 swap_valid、cascade_depth、damage。事件紀錄不得包含玩家密碼、通訊或私人檔案。\n\n## 24｜跨類型：平衡計算示例\nA：若 6 名 60 HP 無護甲敵人與 320 HP、25% 護甲 Boss 都由步槍擊敗且全部命中，理論最少子彈 24+27=51 發。C：5 個音符中 3 Perfect、1 Great、1 Miss，原始分數是 3×1000+600=3600；最後 Combo 因 Miss 歸零，但實際 Combo 需依音符排列決定。D：連鎖第二次下落消除 3 顆紅符文，傷害 round(3×4×1.5)=18。B：存活 185 秒後死亡，結算金幣 floor(185/30)=6。\n\n## 25｜未定案事項與資訊邊界\nA 尚未設計狙擊槍、爆頭倍率、連線模式、掉落子彈的隨機補給。B 尚未定案稀有武器、天賦樹與合作玩法。C 尚未定案長按音符、多鍵音符與全球排行榜。D 尚未定案 PVP 配對、符文炸彈與每日簽到。四個原型都沒有正式上線日期、營運價格或跨平台功能資訊。機制助手被問到未定案項目，應說明「文件未定案」，可以提出明確標示為「設計提案」的替代方案，但不能冒充既定規則。\n\n## 26｜常見錯誤與反例\nA：「Boss 剩 86 HP 就只要補 6 發」是錯的，因護甲使每發步槍實傷 12，至少要 8 次命中。B：「Seek+Separation 就能避開牆壁」是錯的，這不是完整尋路。C：「Combo 倍率讓分數翻倍」是錯的，本原型不使用 Combo 分數倍率。D：「消掉紅色五連會生成炸彈」是錯的，本原型沒有炸彈規則。跨原型：「把 C 的 45ms Perfect 窗口當 A 的射擊判定」是錯的，判定模型不能混用。\n'
SOURCE_PATH.write_text(ORIGINAL_DOCUMENT, encoding='utf-8')
source_text = SOURCE_PATH.read_text(encoding='utf-8')
print('文件路徑：', SOURCE_PATH)
print('文字數：', len(source_text))
print('章節數：', source_text.count('## '))

## 3. 依章節切分 Chunk

用 Markdown 標題定位章節，再把過長段落分成最多 650 字的 Chunk（重疊 100 字）。每一筆保留 `source`、`section` 和 `chunk_id`，之後能在回答中標示來源。


In [ ]:
import re

def split_document(text: str, chunk_size: int = 650, overlap: int = 100):
    assert 0 <= overlap < chunk_size
    sections = re.split(r'(?=^##\s+)', text, flags=re.MULTILINE)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        first_line = section.splitlines()[0].strip().lstrip('#').strip()
        start = 0
        while start < len(section):
            part = section[start:start + chunk_size]
            chunks.append({'chunk_id':len(chunks) + 1,
                           'section':first_line,
                           'source':'game_mechanics_manual.txt',
                           'text':part})
            if start + chunk_size >= len(section):
                break
            start += chunk_size - overlap
    return chunks

chunks = split_document(source_text)
print('切分數：', len(chunks))
for item in chunks[:4]:
    print(item['chunk_id'], item['section'], len(item['text']))
assert len(chunks) >= 16

## 4. Embedding 向量化與建立 FAISS

`multilingual-e5-small` 需使用 `passage:` / `query:` 前綴。若改成 EmbeddingGemma，沿用老師範例 `title: none | text:` / `task: search result | query:`。所有向量先做 L2 正規化，再以 `IndexFlatIP` 比較餘弦相似度。

In [ ]:
import os
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

MODEL_ID = 'intfloat/multilingual-e5-small'
# 如果有 Hugging Face 核准的存取權，可改成：'google/embeddinggemma-300m'
if MODEL_ID == 'google/embeddinggemma-300m':
    from google.colab import userdata
    from huggingface_hub import login
    token = userdata.get('HuggingFace')
    if not token:
        raise RuntimeError('使用 EmbeddingGemma 前，請先在 Colab Secrets 設定 HuggingFace。')
    login(token=token, add_to_git_credential=False)

prefix_type = 'embeddinggemma' if 'embeddinggemma' in MODEL_ID else 'e5'
passage_prefix = 'title: none | text: ' if prefix_type == 'embeddinggemma' else 'passage: '
query_prefix = 'task: search result | query: ' if prefix_type == 'embeddinggemma' else 'query: '
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer(MODEL_ID, device=device)
passages = [passage_prefix + c['text'] for c in chunks]
vectors = embedder.encode(passages, batch_size=8, show_progress_bar=True, convert_to_numpy=True)
vectors = np.ascontiguousarray(vectors, dtype='float32')
faiss.normalize_L2(vectors)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
assert index.ntotal == len(chunks)
print('Embedding 模型：', MODEL_ID)
print('索引向量形狀：', vectors.shape)
print('FAISS 條目：', index.ntotal)

## 5. 在 RAG01 先驗證檢索

這個檢查只進行相似度搜尋，不會呼叫 Groq。

In [ ]:
def search_preview(question, top_k=3):
    qv = embedder.encode([query_prefix + question], convert_to_numpy=True)
    qv = np.ascontiguousarray(qv, dtype='float32')
    faiss.normalize_L2(qv)
    distances, positions = index.search(qv, min(top_k, index.ntotal))
    for score, pos in zip(distances[0], positions[0]):
        if pos >= 0:
            c = chunks[int(pos)]
            print(f"相似度 {score:.3f}｜{c['section']}：{c['text'][:110]}...")

search_preview('《盾線行動》的 Boss 還有 86 HP，至少要命中幾發步槍？')

## 6. 儲存 FAISS + Metadata，封裝成 `rag-data.zip`

ZIP 內含 FAISS index、段落文字、向量模型識別資訊、原始文件。RAG02 只要 ZIP 就能重建檢索（不必重算文件向量）。

In [ ]:
import hashlib
import json
import zipfile
from datetime import datetime, timezone

output_dir = Path('/content/rag-data') if Path('/content').exists() else Path('rag-data')
output_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(output_dir / 'index.faiss'))
(output_dir / 'chunks.json').write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
(output_dir / 'game_mechanics_manual.txt').write_text(source_text, encoding='utf-8')
manifest = {
    'schema_version': 1,
    'model_id': MODEL_ID,
    'prefix_type': prefix_type,
    'index_metric': 'cosine_via_normalized_inner_product',
    'chunk_count': len(chunks),
    'vector_dim': vectors.shape[1],
    'corpus_sha256': hashlib.sha256(source_text.encode('utf-8')).hexdigest(),
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
(output_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = output_dir.parent / 'rag-data.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(output_dir.iterdir()):
        if file.is_file():
            archive.write(file, arcname=file.name)
with zipfile.ZipFile(zip_path) as archive:
    assert set(['index.faiss','chunks.json','manifest.json','game_mechanics_manual.txt']).issubset(archive.namelist())
print('輸出：', zip_path, '大小：', round(zip_path.stat().st_size / 1024, 1), 'KB')
print('請下載後上傳至可公開直接下載的雲端位置。')
if Path('/content').exists():
    from google.colab import files
    files.download(str(zip_path))

## 7. 公開 ZIP 供 RAG02 / 助教讀取

1. 下載上一格輸出的 `rag-data.zip`。
2. 上傳至 Google Drive，將檔案權限改成「知道連結的任何人」可檢視。
3. 複製分享連結，貼到 RAG02 的 `RAG_ZIP_URL`。RAG02 會處理一般 Google Drive 分享連結與公開直接下載網址。
4. 把完整的 RAG02 Colab 檔案分享權限也設為「知道連結的任何人」可檢視。

**請勿上傳 Groq / HuggingFace API key。**